In [5]:
import datasets
import json
import json
import pickle
import re
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from json import JSONDecodeError
from pathlib import Path
from rank_bm25 import BM25Okapi

In [6]:
def save_intermediate(filename, content):
    with open(f"intermediate_states/{filename}.json", "w", encoding="utf-8") as file:
        file.write(json.dumps(content))

def load_intermediate(filename):
    with open(f"intermediate_states/{filename}.json", "r", encoding="utf-8") as file:
        return json.loads(file.read())

Path("intermediate_states").mkdir(parents=True, exist_ok=True)

In [7]:
MODEL_CACHE_DIR = Path(".cache")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def create_agent(name):
    tokenizer = AutoTokenizer.from_pretrained(name, cache_dir=MODEL_CACHE_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        name,
        torch_dtype="auto",
        device_map="auto",
        cache_dir=MODEL_CACHE_DIR,
    )
    agent = {
        "name": name,
        "model": model,
        "tokenizer": tokenizer,
    }
    return agent

agent1 = create_agent("Qwen/Qwen3-0.6B") # information extraction
agent2 = create_agent("HuggingFaceTB/SmolLM2-1.7B-Instruct") # drafting
agent3 = create_agent("LiquidAI/LFM2.5-1.2B-Instruct") # verification
agent4 = create_agent("ibm-granite/granite-3.3-2b-instruct") # revision

Loading weights: 100%|██████████| 218/218 [00:02<00:00, 76.88it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.
Loading weights: 100%|██████████| 148/148 [00:01<00:00, 79.24it/s] 
Some parameters are on the meta device because they were offloaded to the disk and cpu.
Loading weights: 100%|██████████| 362/362 [00:04<00:00, 89.24it/s] 
Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [9]:
def prompt_messages(agent, messages):
    tokenizer = agent["tokenizer"]
    model = agent["model"]

    if agent["name"] == "Qwen/Qwen3-0.6B":
        kwargs = {"enable_thinking": False}
    else:
        kwargs = {}    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        **kwargs,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    inputs_end = inputs["input_ids"].shape[1]
    
    ids = model.generate(
        **inputs,
        max_new_tokens=4096,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
    )
    ids = ids[0][inputs_end:].tolist()
    
    content = tokenizer.decode(ids, skip_special_tokens=True).strip()
    return content

def prompt(agent, system, user):
    messages = []
    messages.append({"role": "system", "content": system})
    messages.append({"role": "user",   "content": user})
    return prompt_messages(agent, messages)

Step 1: Dataset Extraction, Filtering & Clinical Schema Mapping

In [10]:
# Download and filter the dataset, calculate some statistics
def step1():
    def normalize(word):
        word = word.lower()
        word = "".join([c for c in word if c.isalpha()])
        return word

    def get_avg_word_count(notes):
        word_count = [len(note.split()) for note in notes]
        return sum(word_count) / len(word_count)

    def get_avg_len(notes):
        lengths = [len(note) for note in notes]
        return sum(lengths) / len(lengths)

    def get_vocab_size(notes):
        vocab = set()
        for note in notes:
            for word in note.split():
                word = normalize(word)
                vocab.add(word)
        return len(vocab)

    dataset_name = "starmpcc/Asclepius-Synthetic-Clinical-Notes"
    splits = datasets.load_dataset(dataset_name)
    df = splits["train"] # The dataset doesn't have a validation/test set
    df = df.to_pandas()

    # filtering
    df = df[df["task"] == "Summarization"]
    df = df.dropna()
    df = df.drop_duplicates()
    for column in df.columns:
        df = df[df[column] != ""] # drop every row with empty strings

    # sampling
    sample = df.sample(100, random_state=67)

    # calculate statistics
    avg_word_count = get_avg_word_count(df["note"])
    avg_len = get_avg_len(df["note"])
    vocab_size = get_vocab_size(df["note"])
    
    return df, sample, avg_word_count, avg_len, vocab_size

dataset, sample, avg_word_count, avg_len, vocab_size = step1()
notes = sample["note"]

print(f"Average note word count: {avg_word_count:.2f}")
print(f"Average note length:     {avg_len:.2f}")
print(f"Note vocabulary size:    {vocab_size}")

Average note word count: 276.68
Average note length:     1933.85
Note vocabulary size:    58696


Step 2: Build the Information Extraction Agent (Agent 1)

In [ ]:
# Extract the relevant information from the notes into a list of jsons
def step2(notes, agent):
    system_prompt = (
        'You will be given a clinical note in the first user message.\n'
        'You will need to extract relevant information from the clinical note.\n'
        'Relevant information includes primary diagnoses, secondary conditions, critical lab abnormalities, procedures performed, discharge medications and follow up instructions.\n'
        'The output needs to be a valid JSON.\n'
        'If a field has no relevant information, return an empty list [] or an empty string "".\n'
        'Do not give anything other than the valid JSON as output.\n'
        'Do not wrap the JSON in markdown code fences or add any explanation.\n'
        'The output needs to be in following format:\n'
        '{\n'
        '  "primary_diagnoses": [string, ...],\n'
        '  "secondary_conditions": [string, ...],\n'
        '  "critical_lab_abnormalities": [string, ...],\n'
        '  "procedures_performed": [string, ...],\n'
        '  "medications_discharge": [string, ...],\n'
        '  "follow_up_instructions": string\n'
        '}\n'
        'Only include information supported by the note.\n'
        'Do not infer or invent.'
    )

    responses = []
    for i, note in enumerate(notes, 1):
        print(f"{i}/{len(notes)}")
        while True:
            # Prompt on the same note as long as the response is an invalid json
            response = prompt(agent, system_prompt, note)
            try:
                response = json.loads(response)
                break
            except JSONDecodeError:
                continue
        responses.append(response)
    
    return responses

# patient_details = step2(notes, agent1)
# save_intermediate("step2", patient_details) # Save result

1/100
2/100
3/100
4/100
5/100
6/100
7/100
8/100
9/100
10/100
11/100
12/100
13/100


KeyboardInterrupt: 

In [12]:
patient_details = load_intermediate("step2") # Skip step 2 by loading the saved result

Step 3: Implement Single-Agent Baseline

In [ ]:
# Summarize each note into an organized clinical summary similarly to step 2
def step3(notes, agent):
    system_prompt = (
        'You will be given a clinical note in the first user message.\n'
        'You will need to summarize the clinical note using 200-350 words.\n'
        'Write the summary for a clinician, in clinical terminology.\n'
        'You may include relevant information such as reason for encounter, key findings, active problems and plan/follow-up.\n'
        'Do not give anything other than the summary as output.\n'
        'Do not use any markdown in the output or add any explanation.\n'
        'Only include information supported by the note.\n'
        'Do not infer or invent.'
    )
    
    responses = []
    for i, note in enumerate(notes, 1):
        print(f"{i}/{len(notes)}")
        response = prompt(agent, system_prompt, note)
        responses.append(response)
    
    return responses

# summaries = step3(notes, agent1)
# save_intermediate("step3", summaries)

In [13]:
summaries = load_intermediate("step3")

Step 4: Construct Global Evidence Retrieval Index

In [14]:
STEP4_CHUNK_SIZE = 300
STEP4_CHUNK_OVERLAP = 50
STEP4_TOKENIZER_NAME = agent1["name"]
STEP4_TOKENIZER = agent1["tokenizer"]
# chunk notes storage path
STEP4_CHUNK_NOTES_PATH = Path("intermediate_states/step4_chuck_notes.json")
# BM25 index storage path
STEP4_INDEX_PATH = Path("intermediate_states/step4_chuck_index.pkl")
STEP4_CHUNKS_BY_NOTE = {}
STEP4_BM25_INDEX = None

# Converts text to lowercase and extracts word-like terms for BM25.
def tokenize_for_bm25(text):
    return re.findall(r"[a-z0-9]+(?:['-][a-z0-9]+)*", text.lower())

# Divide each clinical note into text chunks of 250–350 tokens with a 50-token overlap between adjacent chunks
def chunk_notes(sample, chunk_size=STEP4_CHUNK_SIZE, overlap=STEP4_CHUNK_OVERLAP):
    if not 250 <= chunk_size <= 350:
        raise ValueError("The project brief requires chunks between 250 and 350 tokens.")
    if not 0 <= overlap < chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size.")

    stride = chunk_size - overlap
    chunks = []
    for row_index, row in sample.iterrows():
        base_id = row.get("source_id", row.get("patient_id", "sample"))
        source_id = f"{base_id}-row-{row_index}"
        note_text = row["note"]
        if not isinstance(note_text, str) or not note_text.strip():
            raise ValueError(f"Note {source_id} must be a non-empty string.")

        token_ids = STEP4_TOKENIZER.encode(note_text, add_special_tokens=False)
        for chunk_number, token_start in enumerate(range(0, len(token_ids), stride)):
            token_end = min(token_start + chunk_size, len(token_ids))
            chunk_text = STEP4_TOKENIZER.decode(token_ids[token_start:token_end], skip_special_tokens=True).strip()
            if chunk_text:
                chunks.append({
                    "source_id": str(source_id),
                    "chunk_id": f"{source_id}-chunk-{chunk_number:03d}",
                    "token_start": token_start,
                    "token_end": token_end,
                    "token_count": token_end - token_start,
                    "tokenizer": STEP4_TOKENIZER_NAME,
                    "text": chunk_text,
                })
            if token_end == len(token_ids):
                break

    save_intermediate("step4_chuck_notes", chunks)
    return chunks

# Index these chunks locally using BM25 (rank-bm25) 
def build_note_indexes():
    if STEP4_CHUNK_NOTES_PATH.exists():
        chunk_records = load_intermediate("step4_chuck_notes")
    else:
        chunk_records = []

    if not chunk_records:
        if "sample" not in globals():
            raise RuntimeError(
                "step4_chuck_notes.json is missing or empty; run Step 1 to regenerate the chunks."
            )
        chunk_records = chunk_notes(sample)

    chunks_by_note = {}
    for chunk in chunk_records:
        source_id = str(chunk["source_id"])
        chunks_by_note.setdefault(source_id, []).append(chunk)

    for note_chunks in chunks_by_note.values():
        note_chunks.sort(key=lambda chunk: chunk["token_start"])

    all_chunks = [
        chunk
        for source_id in sorted(chunks_by_note)
        for chunk in chunks_by_note[source_id]
    ]
    tokenized_corpus = [tokenize_for_bm25(chunk["text"]) for chunk in all_chunks]
    if not tokenized_corpus or any(not tokens for tokens in tokenized_corpus):
        raise ValueError("The global chunk corpus must contain searchable tokens.")

    global_index = BM25Okapi(tokenized_corpus)
    index_record = {
        "index_type": "global_bm25_pickle",
        "tokenizer": STEP4_TOKENIZER_NAME,
        "global_index": global_index,
        "chunks": all_chunks,
    }
    with STEP4_INDEX_PATH.open("wb") as index_file:
        pickle.dump(index_record, index_file, protocol=pickle.HIGHEST_PROTOCOL)
    return chunks_by_note, global_index, all_chunks


STEP4_CHUNKS_BY_NOTE, STEP4_BM25_INDEX, step4_all_chunks = build_note_indexes()

print("Indexed notes:", len(STEP4_CHUNKS_BY_NOTE))
print("Total chunks:", len(step4_all_chunks))
print("Saved chunk notes:", STEP4_CHUNK_NOTES_PATH)
print("Saved serialized global BM25 index:", STEP4_INDEX_PATH)

Indexed notes: 100
Total chunks: 203
Saved chunk notes: intermediate_states\step4_chuck_notes.json
Saved serialized global BM25 index: intermediate_states\step4_chuck_index.pkl


In [ ]:
# a helper function: given a statement from a summary, search the source note 
# and return the top_k most relevant text chunks
def retrieve_evidence(statement, top_k=2):
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")
    if not STEP4_INDEX_PATH.exists():
        raise FileNotFoundError(f"Build the Step 4 index first: {STEP4_INDEX_PATH}")
    query_terms = tokenize_for_bm25(statement)
    if not query_terms:
        raise ValueError("statement must contain at least one searchable token.")

    with STEP4_INDEX_PATH.open("rb") as index_file:
        index_record = pickle.load(index_file)
    if index_record.get("index_type") != "global_bm25_pickle":
        raise ValueError("The saved Step 4 file does not contain a serialized global BM25 index.")

    global_index = index_record["global_index"]
    scores = global_index.get_scores(query_terms)
    chunks = index_record["chunks"]
    ranked_indices = sorted(range(len(scores)), key=lambda index: (-scores[index], index))[:top_k]
    return [
        {**chunks[index], "bm25_score": float(scores[index])}
        for index in ranked_indices
    ]

# Update sample statement to retrieve top_k most relevant chunks
sample_statement = 'The patient was treated with high dose aspirin, IVIG, and infliximab as per the hospital protocol. Follow-up appointments with Cardiology and PCP were scheduled'
sample_results = retrieve_evidence(sample_statement, top_k=2)

print("Global search returned", len(sample_results), "chunks from",
      sorted({result["source_id"] for result in sample_results}))
for rank, result in enumerate(sample_results, start=1):
    print(f"\nRelevant chunk {rank}:", result["source_id"])
    print("BM25 score:", f"{result['bm25_score']:.4f}")
    print(result["text"])

Step 5: Assemble Multi-Agent Workflow (Generation, Verification & Revision)

In [15]:
# Write an initial clinical summary based on the information extracted by the first agent
def step5_draft_summary(patient_details, agent):
    def flatten(details):
        primary_diagnoses          = ", ".join(details["primary_diagnoses"]) if "primary_diagnoses" in details else ""
        secondary_conditions       = ", ".join(details["secondary_conditions"]) if "secondary_conditions" in details else ""
        critical_lab_abnormalities = ", ".join(details["critical_lab_abnormalities"]) if "critical_lab_abnormalities" in details else ""
        procedures_performed       = ", ".join(details["procedures_performed"]) if "procedures_performed" in details else ""
        medications_discharge      = ", ".join(details["medications_discharge"]) if "medications_discharge" in details else ""
        follow_up_instructions     = details["follow_up_instructions"] if "follow_up_instructions" in details else ""
        return (
            '{\n'
            f'  "primary_diagnoses": [{primary_diagnoses}],\n'
            f'  "secondary_conditions": [{secondary_conditions}],\n'
            f'  "critical_lab_abnormalities": [{critical_lab_abnormalities}],\n'
            f'  "procedures_performed": [{procedures_performed}],\n'
            f'  "medications_discharge": [{medications_discharge}],\n'
            f'  "follow_up_instructions": {follow_up_instructions}\n'
            '}'
        )

    system_prompt = (
        'You will be given key patient details extracted from a clinical note in the first user message.\n'
        'You will need to write an initial clinical summary based on the key patient details using 200-350 words.\n'
        'The input will be in the following format:\n'
        '{\n'
        '  "primary_diagnoses": [string, ...],\n'
        '  "secondary_conditions": [string, ...],\n'
        '  "critical_lab_abnormalities": [string, ...],\n'
        '  "procedures_performed": [string, ...],\n'
        '  "medications_discharge": [string, ...],\n'
        '  "follow_up_instructions": string\n'
        '}\n'
        'Write the summary for a clinician, in clinical terminology.\n'
        'You may include relevant information such as reason for encounter, key findings, active problems and plan/follow-up.\n'
        'Do not give anything other than the summary as output.\n'
        'Do not use any markdown in the output or add any explanation.\n'
        'Only include information supported by the patient details.\n'
        'Do not infer or invent.'
    )
    
    responses = []
    for i, details in enumerate(patient_details, 1):
        print(f"{i}/{len(patient_details)}")
        response = prompt(agent, system_prompt, flatten(details))
        responses.append(response)
    
    return responses

In [16]:
# Write factual statements based on the summary, expand abbreviations and categorize each statement using the source note
def step5_draft_statement(summaries, notes, agent):
    def expand(statements):
        lines = statements.split("\n")
        lines = [[dictionary[word.lower()] if word.lower() in dictionary else word for word in line.split()] for line in lines]
        lines = "\n".join([" ".join(line) for line in lines])
        return lines
        
    def get_dictionary():
        with open("dictionary.txt", "r", encoding="utf-8") as file:
            dictionary = file.read()
        dictionary = dictionary.strip().split("\n")
        dictionary = [(dictionary[i], dictionary[i+1]) for i in range(0, len(dictionary), 2)]
        dictionary = [(k.lower(), v) for k, v in dictionary]
        for k, v in dictionary:
            if "." in k:
                dictionary.append((k.replace(".", ""), v))
        dictionary = dict(dictionary)
        return dictionary
    
    system_prompt = (
        'You will be given some instructions in each user message.\n'
        'Your job is to follow the instructions and only return what the instructions tell you to return.\n'
        'Do not give anything other than what the instructions ask for as output.\n'
        'Do not use any markdown in the output or add any explanation.\n'
        'Only include information supported by the input.\n'
        'Do not infer or invent.'
    )

    prompt1 = (
        'You will be given a draft summary of a clinical note in the next user message.\n'
        'You will need to break the draft summary into individual factual statements.\n'
        'The output will need to be in the following format:\n'
        '1. [STATEMENT]\n'
        '2. [STATEMENT]\n'
        '3. [STATEMENT]\n'
        '...\n'
        'Where each [STATEMENT] is an individual factual statement.\n'
        'Write the statements for a clinician, in clinical terminology.\n'
        'Do not give anything other than the statements as output.\n'
        'Do not use any markdown in the output or add any explanation.\n'
        'Only include information supported by the summary.\n'
        'Do not infer or invent.'
    )

    prompt2 = (
        'You will be given the source note from which the draft summary was built in the next user message.\n'
        'You will need to categorize each statement into one of the following categories:\n'
        'Supported, Unsupported, Contradictory or Omitted Critical Finding.\n'
        'The output will need to be in the following format:\n'
        '1. [CATEGORY] [STATEMENT]\n'
        '2. [CATEGORY] [STATEMENT]\n'
        '3. [CATEGORY] [STATEMENT]\n'
        '...\n'
        'Where each [STATEMENT] is an individual factual statement you extracted previously and [CATEGORY] is one of the four categories surrounded by square brackets.\n'
        'You need to base the categorization on evidence gathered from the source note.\n'
        'Do not give anything other than the categories and the statements as output.\n'
        'Do not use any markdown in the output or add any explanation.\n'
        'Do not infer or invent.'
    )
    
    dictionary = get_dictionary()
    responses = []
    for i, (summary, note) in enumerate(zip(summaries, notes), 1):
        print(f"{i}/{len(summaries)}")
        messages = []
        messages.append({"role": "system",    "content": system_prompt})
        messages.append({"role": "user",      "content": prompt1})
        messages.append({"role": "user",      "content": summary})
        response = prompt_messages(agent, messages)
        response = expand(response)
        
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user",      "content": prompt2})
        messages.append({"role": "user",      "content": note})
        response = prompt_messages(agent, messages)
        
        responses.append(response)
    
    return responses

In [17]:
# Verifying each statement using the draft summaries gathered previously. Remove or modify incorrect statements.
def step5_revision(summaries, statements, agent):
    system_prompt = (
        'You will be given a summary of a clinical note in the first user message and a list of factual statements in the second user message.\n'
        'You will need to revise the summary by rewriting or removing unsupported claims according to the list of factual statements.\n'
        'Each factual statement has already been categorized into one of the following categories:\n'
        'Supported, Unsupported, Contradictory or Omitted Critical Finding.\n'
        'The factual statements will be in the following format:\n'
        '1. [CATEGORY] [STATEMENT]\n'
        '2. [CATEGORY] [STATEMENT]\n'
        '3. [CATEGORY] [STATEMENT]\n'
        '...\n'
        'Where each [STATEMENT] is an individual statement and [CATEGORY] is one of the four categories surrounded by square brackets.\n'
        'Do not give anything other than the revised summary as output.\n'
        'Do not use any markdown in the output or add any explanation.\n'
        'Do not infer or invent.'
    )
    
    responses = []
    for i, (summary, statement) in enumerate(zip(summaries, statements), 1):
        print(f"{i}/{len(summaries)}")
        messages = []
        messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user",   "content": summary})
        messages.append({"role": "user",   "content": statement})
        response = prompt_messages(agent, messages)
        responses.append(response)
    
    return responses

Step 6: Pipeline Execution & Intermediate State TracingStep 

In [ ]:
# draft_summaries = step5_draft_summary(patient_details, agent2)
# save_intermediate("step5_draft_summaries", draft_summaries)

# draft_statements = step5_draft_statement(draft_summaries, notes, agent3)
# save_intermediate("step5_draft_statements", draft_statements)

# revised_summaries = step5_revision(draft_summaries, draft_statements, agent4)
# save_intermediate("step5_revised_summaries", revised_summaries)

In [18]:
draft_summaries = load_intermediate("step5_draft_summaries")
draft_statements = load_intermediate("step5_draft_statements")
revised_summaries = load_intermediate("step5_revised_summaries")

In [19]:
logs = []
for i, (summary, draft_summary, statement, revised_summary) in enumerate(zip(summaries, draft_summaries, draft_statements, revised_summaries), 1):
    row = {
        "id": i,
        "summary": {
            "baseline": summary,
            "draft": draft_summary,
            "revised": revised_summary,
        },
        "statements": statement,
    }
    logs.append(row)

Step 7: Automated Evaluation (ROUGE, BERTScore, METEOR)

Install the evaluation dependencies once with `%pip install -r requirements-step7.txt` if needed.

### 7.1 Prepare aligned evaluation inputs
Validate all inputs before scoring; do not silently truncate mismatched lists.

In [20]:
def step7_prepare_inputs(sample, summaries, draft_summaries, revised_summaries, expected_cases=100):
    if not {"note", "answer"}.issubset(sample.columns):
        raise ValueError("The Step 1 sample must contain note and answer columns.")
    if len(sample) != expected_cases or not sample.index.is_unique:
        raise ValueError(f"Expected {expected_cases} cases with unique dataset indices.")
    variants = {
        "baseline": summaries,
        "draft": draft_summaries,
        "revised": revised_summaries,
    }
    texts = {"note": sample["note"].tolist(), "answer": sample["answer"].tolist(), **variants}
    for label, values in texts.items():
        if not isinstance(values, (list, tuple)) or len(values) != expected_cases:
            raise ValueError(f"{label} must contain exactly {expected_cases} texts.")
        for position, text in enumerate(values, 1):
            if not isinstance(text, str) or not text.strip():
                raise ValueError(f"{label}, case {position}: expected a non-empty string.")
    rows = []
    for position, (dataset_index, case) in enumerate(sample.iterrows()):
        for variant, predictions in variants.items():
            rows.append({
                "case_id": position + 1,
                "dataset_index": str(dataset_index),
                "variant": variant,
                "note": case["note"],
                "reference": case["answer"],
                "prediction": predictions[position],
            })
    return pd.DataFrame(rows)


### 7.2 Calculate automatic metrics
Each metric has its own function. ROUGE reports stemmed F1; METEOR uses lowercase word/punctuation tokens and WordNet synonyms.
BERTScore reports unrescaled F1 with `distilbert-base-uncased`, layer 6, no IDF weighting, and a shared encoder for all variants.
These BERTScore values should only be compared with runs using the same settings; inputs beyond the encoder's token limit are truncated by BERTScore.
New model and WordNet downloads are cached under the project's `.cache` directory; an existing WordNet installation can be reused.
The first evaluation requires internet access unless the resources are already cached.

In [22]:
def step7_calculate_rouge(evaluation_inputs):
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rows = []
    for case in evaluation_inputs.itertuples(index=False):
        scores = scorer.score(case.reference, case.prediction)
        rows.append({name: score.fmeasure for name, score in scores.items()})
    return pd.DataFrame(rows, index=evaluation_inputs.index)


def step7_calculate_meteor(evaluation_inputs, cache_dir=Path(".cache")):
    import nltk
    from nltk.corpus import wordnet
    from nltk.tokenize import wordpunct_tokenize
    from nltk.translate.meteor_score import meteor_score

    nltk_dir = str((Path(cache_dir) / "nltk").resolve())
    Path(nltk_dir).mkdir(parents=True, exist_ok=True)
    if nltk_dir not in nltk.data.path:
        nltk.data.path.insert(0, nltk_dir)
    try:
        wordnet.ensure_loaded()
    except LookupError:
        nltk.download("wordnet", download_dir=nltk_dir, raise_on_error=True)
        wordnet.ensure_loaded()
    scores = [
        meteor_score(
            [wordpunct_tokenize(case.reference.lower())],
            wordpunct_tokenize(case.prediction.lower()),
        )
        for case in evaluation_inputs.itertuples(index=False)
    ]
    return pd.DataFrame({"meteor": scores}, index=evaluation_inputs.index)


def step7_calculate_bertscore(evaluation_inputs, cache_dir=Path(".cache"), batch_size=8):
    from bert_score import BERTScorer
    from huggingface_hub import snapshot_download

    if batch_size < 1:
        raise ValueError("BERTScore batch_size must be at least 1.")
    model_path = snapshot_download(
        "distilbert/distilbert-base-uncased",
        cache_dir=str(cache_dir),
        allow_patterns=["*.json", "*.txt", "*.safetensors"],
    )
    scorer = BERTScorer(
        model_type=model_path,
        num_layers=6,
        batch_size=batch_size,
        device="cpu",
        idf=False,
        rescale_with_baseline=False,
        use_fast_tokenizer=True,
    )
    _, _, f1 = scorer.score(
        evaluation_inputs["prediction"].tolist(),
        evaluation_inputs["reference"].tolist(),
        batch_size=batch_size,
    )
    return pd.DataFrame({"bertscore_f1": f1.cpu().tolist()}, index=evaluation_inputs.index)


### 7.3 Measure summary length and compression
Word counts use whitespace splitting. Compression ratio = summary words / source-note words (lower means shorter).
Also record summary/reference length ratios and paired revised-minus-draft word changes.
The revised/draft word ratio and fraction of cases that grew expose length inflation; they do not by themselves establish factual quality.

In [23]:
def step7_calculate_lengths(evaluation_inputs):
    lengths = pd.DataFrame(index=evaluation_inputs.index)
    lengths["summary_words"] = evaluation_inputs["prediction"].str.split().str.len()
    lengths["reference_words"] = evaluation_inputs["reference"].str.split().str.len()
    lengths["source_words"] = evaluation_inputs["note"].str.split().str.len()
    lengths["compression_ratio"] = lengths["summary_words"] / lengths["source_words"]
    lengths["summary_reference_ratio"] = lengths["summary_words"] / lengths["reference_words"]
    return lengths


def step7_compare_revision_lengths(per_case):
    word_counts = per_case.pivot(index="case_id", columns="variant", values="summary_words")
    comparison = word_counts.rename(columns={
        "baseline": "baseline_words", "draft": "draft_words", "revised": "revised_words",
    }).reset_index()
    comparison.columns.name = None
    comparison["revised_minus_draft_words"] = comparison["revised_words"] - comparison["draft_words"]
    comparison["revised_draft_word_ratio"] = comparison["revised_words"] / comparison["draft_words"]
    comparison["revision_is_longer"] = comparison["revised_words"] > comparison["draft_words"]
    return comparison


### 7.4 Run evaluation, aggregate, and save
Scores are on a 0-1 scale (BERTScore can be negative for poor matches). Overall scores are arithmetic means of per-case values.
Save per-case results, variant-level means, revision-length comparisons, and settings using the existing intermediate-state helper.
The evaluation is CPU-based so it does not compete with the four agents for GPU memory.

In [24]:
def step7_evaluate(sample, summaries, draft_summaries, revised_summaries, expected_cases=100):
    evaluation_inputs = step7_prepare_inputs(
        sample, summaries, draft_summaries, revised_summaries, expected_cases,
    )
    per_case = pd.concat([
        evaluation_inputs[["case_id", "dataset_index", "variant"]],
        step7_calculate_rouge(evaluation_inputs),
        step7_calculate_meteor(evaluation_inputs),
        step7_calculate_bertscore(evaluation_inputs),
        step7_calculate_lengths(evaluation_inputs),
    ], axis=1)
    numeric_columns = [
        "rouge1", "rouge2", "rougeL", "meteor", "bertscore_f1",
        "summary_words", "reference_words", "source_words",
        "compression_ratio", "summary_reference_ratio",
    ]
    if per_case[numeric_columns].isna().any().any():
        raise ValueError("Evaluation produced missing scores or length statistics.")
    aggregate = per_case.groupby("variant", sort=False)[numeric_columns].mean().reset_index()
    aggregate.insert(1, "case_count", expected_cases)
    revision_lengths = step7_compare_revision_lengths(per_case)
    return per_case, aggregate, revision_lengths


step7_per_case, step7_aggregate, step7_revision_lengths = step7_evaluate(
    sample, summaries, draft_summaries, revised_summaries,
)
save_intermediate("step7_per_case", step7_per_case.to_dict(orient="records"))
save_intermediate("step7_aggregate", step7_aggregate.to_dict(orient="records"))
save_intermediate("step7_revision_lengths", step7_revision_lengths.to_dict(orient="records"))
save_intermediate("step7_settings", {
    "reference_column": "answer",
    "rouge": "F1, stemming enabled",
    "meteor_tokenizer": "lowercase wordpunct_tokenize",
    "bertscore_model": "distilbert/distilbert-base-uncased",
    "bertscore_num_layers": 6,
    "bertscore_idf": False,
    "bertscore_rescale_with_baseline": False,
    "compression_ratio": "summary_words / source_words",
    "alignment": "Positional; summaries must use the Step 1 sample order.",
})
print(step7_aggregate.to_string(index=False))
print("Mean revised/draft word ratio:", step7_revision_lengths["revised_draft_word_ratio"].mean())
print("Mean revised-minus-draft words:", step7_revision_lengths["revised_minus_draft_words"].mean())
print("Fraction of revisions longer than drafts:", step7_revision_lengths["revision_is_longer"].mean())


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2096.80it/s]
[transformers] DistilBertModel LOAD REPORT from: D:\UNIBZ\FinlandExchange\Course\NaturalLanguage\FinalProject\.cache\models--distilbert--distilbert-base-uncased\snapshots\12040accade4e8a0f71eabdb258fecc2e7e948be
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 variant  case_count   rouge1   rouge2   rougeL   meteor  bertscore_f1  summary_words  reference_words  source_words  compression_ratio  summary_reference_ratio
baseline         100 0.486381 0.276922 0.351271 0.532125      0.814945         148.68            74.84        276.71           0.562751                 2.138958
   draft         100 0.418316 0.211695 0.301580 0.345767      0.798604          60.34            74.84        276.71           0.233507                 0.888052
 revised         100 0.461989 0.212921 0.311843 0.418304      0.802780          92.57            74.84        276.71           0.352962                 1.339674
Mean revised/draft word ratio: 1.6636150548703414
Mean revised-minus-draft words: 32.23
Fraction of revisions longer than drafts: 0.9


Step 8: Manual Verification & Factuality Audit

Step 9: Agent Ablation & Resource Analysis

Step 10: Results Visualization (Charts & Figures)

Step 11: Synthesis, Literature Review & Final Reporting